# Ultimate AI Stack
## Multi-Model Orchestrator: Gemini + Claude + OpenAI

Intelligent routing between AI models based on task type, cost, and quality.

> **Note:** Add your API keys to the `.env` file to activate.

---

In [ ]:
# CELL 1: Setup
import subprocess, sys
for p in ['python-dotenv','requests','tiktoken','aiohttp']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', p])

import os, json, time, hashlib, re
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional, Dict, List
from dotenv import load_dotenv
load_dotenv()
OUTPUT_DIR = Path('outputs'); OUTPUT_DIR.mkdir(exist_ok=True)
print('AI Stack Ready!')

In [ ]:
# CELL 2: Multi-Model Client
@dataclass
class ModelConfig:
    name: str
    provider: str        # 'gemini', 'anthropic', 'openai'
    api_key_env: str     # Environment variable name for API key
    max_tokens: int = 4096
    cost_per_1k_input: float = 0.0     # USD per 1K input tokens
    cost_per_1k_output: float = 0.0    # USD per 1K output tokens
    strengths: List[str] = field(default_factory=list)

# Model registry
MODELS = {
    'gemini-pro': ModelConfig('gemini-2.0-flash', 'gemini', 'GEMINI_API_KEY',
        cost_per_1k_input=0.00035, cost_per_1k_output=0.0015,
        strengths=['multimodal', 'long_context', 'reasoning', 'search']),
    'claude-sonnet': ModelConfig('claude-sonnet-4-20250514', 'anthropic', 'ANTHROPIC_API_KEY',
        cost_per_1k_input=0.003, cost_per_1k_output=0.015,
        strengths=['coding', 'analysis', 'safety', 'instruction_following']),
    'gpt-4': ModelConfig('gpt-4o', 'openai', 'OPENAI_API_KEY',
        cost_per_1k_input=0.005, cost_per_1k_output=0.015,
        strengths=['general', 'creative', 'reasoning']),
}

class AIOrchestrator:
    """Routes tasks to the best AI model based on task type and availability."""
    
    def __init__(self):
        self.models = MODELS
        self.history = []
        self.total_cost = 0.0
        self._check_api_keys()
    
    def _check_api_keys(self):
        self.available = {}
        for name, cfg in self.models.items():
            key = os.environ.get(cfg.api_key_env, '')
            self.available[name] = bool(key)
            status = 'ACTIVE' if key else 'NO KEY'
            print(f"  {name}: {status}")
    
    def route(self, task_type: str) -> str:
        """Select best model for task type."""
        task_mapping = {
            'code': 'claude-sonnet', 'analysis': 'claude-sonnet',
            'creative': 'gpt-4', 'multimodal': 'gemini-pro',
            'search': 'gemini-pro', 'general': 'gemini-pro',
            'reasoning': 'claude-sonnet', 'math': 'gemini-pro',
        }
        preferred = task_mapping.get(task_type, 'gemini-pro')
        if self.available.get(preferred): return preferred
        # Fallback to any available
        for name, avail in self.available.items():
            if avail: return name
        return preferred  # Return anyway for demo
    
    def estimate_cost(self, model_name, input_tokens, output_tokens):
        cfg = self.models[model_name]
        return (cfg.cost_per_1k_input * input_tokens / 1000 +
                cfg.cost_per_1k_output * output_tokens / 1000)
    
    def query(self, prompt, task_type='general'):
        """Route and query the best model. Returns simulated response if no API key."""
        model = self.route(task_type)
        cost = self.estimate_cost(model, len(prompt.split())*1.3, 500)
        self.total_cost += cost
        self.history.append({'model': model, 'task': task_type, 'cost': cost, 'time': time.time()})
        
        # In production, this would call the actual API
        return {
            'model': model, 'task_type': task_type,
            'estimated_cost': f'${cost:.4f}',
            'message': f'[{model}] would process: "{prompt[:50]}..."',
            'note': 'Add API keys to .env for real responses'
        }

orch = AIOrchestrator()
print("\nRouting tests:")
for task in ['code', 'creative', 'search', 'math', 'analysis']:
    r = orch.query(f"Test {task} query", task)
    print(f"  {task} -> {r['model']} (est. {r['estimated_cost']})")

In [ ]:
# CELL 3: Create .env template
env_template = '''# === Ultimate AI Stack Configuration ===
# Add your API keys below (never commit this file!)

# Google Gemini API
# Get yours at: https://aistudio.google.com/apikey
GEMINI_API_KEY=

# Anthropic Claude API
# Get yours at: https://console.anthropic.com/
ANTHROPIC_API_KEY=

# OpenAI API
# Get yours at: https://platform.openai.com/api-keys
OPENAI_API_KEY=
'''

env_path = Path('.env')
if not env_path.exists():
    env_path.write_text(env_template)
    print("Created .env template - add your API keys!")
else:
    print(".env already exists")

print("\nP10 Ultimate AI Stack - COMPLETE!")
print("Add API keys to .env and re-run for live queries.")